# Day 88 — Validation

This notebook covers Held-Out Model Evaluation (`evaluate`, `fit_with_evaluation`), Inference Hygiene (`model.eval()` + `torch.no_grad()`), Regression Metrics (`MSE`, `RMSE`, `MAE`, $R^2$), and Conditional Checkpoint Saving (`model.state_dict()`)

## Validation goals
- Combine `model.eval()` and `with torch.no_grad():` in an `evaluate` helper to compute deterministic, graph-free validation metrics
- Implement `MSE`, `RMSE`, `MAE`, and coefficient of determination ($R^2$) across full evaluation datasets using sample-weighted averaging
- Track training and held-out validation metrics across 30 epochs in `fit_with_evaluation` and save the best checkpoint (`model.state_dict()`) conditionally when `val_rmse` improves
- Diagnose train vs. validation error curves (healthy convergence, overfitting, and underfitting) and identify remediation strategies

### 1. Why Evaluation Needs Its Own Mode and Context

Running evaluation requires two safeguards that differ from training:
- `model.eval()`: Switches layers with training-specific behavior—such as nn.Dropout—into inference mode so predictions are deterministic and full network capacity is used.
- `torch.no_grad()`: Disables autograd graph construction during forward passes on validation/test batches, saving memory and speeding up evaluation.

Note that `model.eval()` does not automatically disable gradients on its own, and with `torch.no_grad()`: does not switch dropout off—using both together is standard practice.

### 2. Implementing Evaluation & Joint Train/Val Tracking (`day_88_validation.py`)

Let's write an evaluate helper that computes `MSE`, `RMSE`, `MAE`, and $R^2$ over any DataLoader, and combine it with your train_one_epoch helper from Day 87 to log both training and held-out metrics across epochs.

In [1]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from day_85_dataset import build_dataloaders
from day_86_architecture import RetailDemandMLP
from day_87_training import set_seed, train_one_epoch


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    """Evaluates model on a DataLoader without gradient tracking."""
    model.eval()
    total_mse_sum = 0.0
    total_mae_sum = 0.0
    total_samples = 0

    all_preds: list[np.ndarray] = []
    all_targets: list[np.ndarray] = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_size = batch_x.shape[0]

            preds = model(batch_x)
            loss = criterion(preds, batch_y)

            total_mse_sum += loss.item() * batch_size
            total_mae_sum += torch.abs(preds - batch_y).sum().item()
            total_samples += batch_size

            all_preds.append(preds.squeeze(-1).cpu().numpy())
            all_targets.append(batch_y.squeeze(-1).cpu().numpy())

    mse = total_mse_sum / total_samples
    rmse = float(np.sqrt(mse))
    mae = total_mae_sum / total_samples

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0.0 else 0.0

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


def fit_with_evaluation(
    model: nn.Module,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int = 30,
    log_every: int = 5,
    checkpoint_path: str | None = None,
) -> list[dict[str, float]]:
    """Trains across epochs and tracks held-out metrics each epoch."""
    history: list[dict[str, float]] = []
    best_val_rmse = float("inf")

    for epoch in range(1, epochs + 1):
        train_mse, train_mae = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )
        train_rmse = float(np.sqrt(train_mse))

        val_metrics = evaluate(
            model=model,
            dataloader=eval_loader,
            criterion=criterion,
            device=device,
        )

        if val_metrics["rmse"] < best_val_rmse and checkpoint_path is not None:
            best_val_rmse = val_metrics["rmse"]
            Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)

        row = {
            "epoch": float(epoch),
            "train_rmse": train_rmse,
            "train_mae": train_mae,
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_r2": val_metrics["r2"],
        }
        history.append(row)

        if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:2d}/{epochs:2d} | "
                f"Train RMSE: {train_rmse:6.2f} | "
                f"Val RMSE: {val_metrics['rmse']:6.2f} | "
                f"Val MAE: {val_metrics['mae']:6.2f} | "
                f"Val R2: {val_metrics['r2']:6.4f}"
            )

    return history


if __name__ == "__main__":
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, test_loader, num_features, source = build_dataloaders(batch_size=32)
    model = RetailDemandMLP(
        in_features=num_features,
        hidden_dims=(64, 32),
        dropout=0.1,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

    print("--- Day 88: Training with Held-Out Evaluation ---")
    print(f"Data source: {source} | Device: {device} | Features: {num_features}\n")

    history = fit_with_evaluation(
        model=model,
        train_loader=train_loader,
        eval_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=30,
        log_every=5,
        checkpoint_path="checkpoints/retail_demand_mlp_best.pt",
    )

--- Day 88: Training with Held-Out Evaluation ---
Data source: synthetic_fallback | Device: cpu | Features: 12

Epoch  1/30 | Train RMSE: 151.09 | Val RMSE:  84.88 | Val MAE:  81.83 | Val R2: -8.0900
Epoch  5/30 | Train RMSE:  21.39 | Val RMSE:  16.98 | Val MAE:  13.52 | Val R2: 0.6364
Epoch 10/30 | Train RMSE:  21.83 | Val RMSE:  15.92 | Val MAE:  12.85 | Val R2: 0.6803
Epoch 15/30 | Train RMSE:  20.66 | Val RMSE:  15.35 | Val MAE:  12.52 | Val R2: 0.7026
Epoch 20/30 | Train RMSE:  21.98 | Val RMSE:  16.69 | Val MAE:  13.71 | Val R2: 0.6484
Epoch 25/30 | Train RMSE:  21.71 | Val RMSE:  15.59 | Val MAE:  12.83 | Val R2: 0.6932
Epoch 30/30 | Train RMSE:  22.06 | Val RMSE:  15.37 | Val MAE:  12.67 | Val R2: 0.7019


### 3. Reading Train vs. Validation Curves

Comparing training and evaluation metrics side by side highlights three common patterns:
- Healthy Convergence: Both Train `RMSE` and `Val RMSE` drop and stabilize close together, with `Val R2` rising steadily.
- Overfitting: Train RMSE keeps falling while `Val RMSE` flattens out or starts climbing—a sign to lower learning rate, add regularization/dropout, simplify hidden width, or stop earlier.
- Underfitting: Both Train RMSE and `Val RMSE` stay high—often pointing to an overly small network, too few epochs, under-tuned learning rate, or uninformative/unscaled features.

### Guided Question:

Why do we combine `model.eval()` with with `torch.no_grad()`: during evaluation instead of relying on only one of them?

### Answer:

Combining `model.eval()` with with `torch.no_grad()`: during evaluation is necessary because they control two separate mechanisms in PyTorch: `model.eval()` changes layer behavior (such as dropout and batch normalization), while `torch.no_grad()` disables gradient tracking and autograd graph construction.

**What `model.eval()` Controls (Layer Mode)**

- Switches modules to inference mode: Layers like `nn.Dropout` stop dropping activations, and `nn.BatchNorm` layers stop updating running mean/variance statistics and use their stored running estimates instead of batch statistics.
- Does NOT disable autograd: Even under `model.eval()`, PyTorch still tracks operations and builds computation graphs if gradients are enabled, which wastes memory and compute during evaluation.

**What with `torch.no_grad()`: Controls (Autograd Engine)**

- Skips gradient recording: Operations inside the block do not build a backward graph or require gradient buffers, cutting memory usage and speeding up forward passes.
- Does NOT change layer behavior: Dropout still randomly zeroes units and batch normalization still updates running stats if the model is still in `model.train()` mode.

**Why You Need Both Together**

- Only `model.eval()`: Predictions and layer statistics behave properly for inference, but PyTorch still allocates memory for the autograd graph unnecessarily.
- Only `torch.no_grad()`: Memory and speed improve because gradients are off, but nn.Dropout remains active and nn.BatchNorm still mutates running statistics using evaluation batches.
- Both `model.eval()` and with `torch.no_grad()`: Evaluation is deterministic, does not leak or alter running normalization state, and runs with lower memory overhead and faster execution.

### Guided Question:

What divergence between Train `RMSE` and `Val RMSE` across epochs indicates overfitting, and what adjustments can you test when it happens?

### Answer:

Overfitting shows up when Train RMSE keeps decreasing across epochs while Val RMSE levels off, rises, or stays noticeably higher than Train RMSE—creating a widening gap between training fit and unseen-data generalization.

**How to Read Train `RMSE` vs.  Curves**

- Healthy convergence: Both Train RMSE and Val RMSE drop across epochs and converge to a relatively small, stable gap.
- Classic overfitting: Train RMSE continues falling smoothly while Val RMSE stops improving early and begins climbing upward (a U-shaped validation curve) or plateaus well above Train RMSE.
- Underfitting (contrast): Both Train RMSE and Val RMSE stay high or plateau together with little gap, meaning the model lacks capacity or informative features rather than memorizing training rows.
- Noisy validation spikes: Brief, irregular bumps in Val RMSE can happen on small validation sets or with high learning rates, so look at the multi-epoch trend rather than a single epoch.

**Adjustments to Test When Overfitting Appears**

- Add or increase regularization: Raise `weight_decay` in your optimizer (such as AdamW) or add/increase `nn.Dropout` rates in hidden layers so weights cannot rely on brittle feature combinations or training noise.
- Reduce model capacity: Narrow hidden layer widths, remove a hidden layer, or simplify input features if the network has too many parameters relative to your dataset size.
- Use early stopping: Track Val RMSE after each epoch, save the checkpoint at its lowest validation error, and stop training once Val RMSE fails to improve for a set patience window.
- Check data split and leakage: Verify that validation rows do not leak future information, target-derived features, or duplicate/near-duplicate store-SKU records from the training set, and ensure temporal splits match how predictions are made in production.
- Tune learning rate and batch size: If training loss plunges rapidly while validation rises, try a slightly smaller learning rate or adjust batch size so updates do not overfit early noise.

### Guided Question:

Why is saving `model.state_dict()` only when val_rmse improves useful for longer training runs?

### Answer:

Saving `model.state_dict()` only when val_rmse improves keeps your best-generalizing checkpoint on disk without wasting time or storage saving inferior epochs during a longer training run.

**Key Benefits During Longer Runs**

- Captures the sweet spot before overfitting: In long runs, validation error often bottoms out at some intermediate epoch and then rises as the model overfits. Saving only on improvement preserves that best weights snapshot automatically instead of leaving you with a degraded final-epoch model.
- Saves disk space and I/O time: Writing checkpoints every epoch on large models or slow storage adds overhead and fills disk space with worse weights. Conditional saving writes only when a genuine gain occurs.
- Provides a reliable baseline for rollback or post-training: Once training finishes (or hits early stopping), you can immediately reload the best state_dict for test evaluation, inference, or fine-tuning without manually scanning logs to guess which epoch was strongest.